# Mão na Roda — Pipeline Integrado (BERTimbau + Redirect + Base Canônica + RAG)

**Universidade Presbiteriana Mackenzie**  
Faculdade de Computação e Informática  
Disciplina: Inteligência Artificial — 7ºN CC (Noite)  
Professor: Prof. Dr. Ivan Carlos Alcântara de Oliveira

---

**Projeto:** Mão na Roda — Sistema Inteligente para Diagnóstico Preliminar de Falhas Automotivas Baseado em PLN e Machine Learning

| Integrante | RA | E-mail |
|---|---|---|
| Diego Spagnuolo Sugai | 10417329 | diegossugai@gmail.com |
| Kauê Henrique Matias Alves | 10417894 | kaueh.malves@gmail.com |
| Leonardo Moreira dos Santos | 10417555 | leonardomsantos12@gmail.com |
| Victor Maki Tarcha | 10419861 | victormakitarcha@gmail.com |

---

## Objetivo deste notebook

Implementar o pipeline **end-to-end** que integra os quatro artefatos do projeto:

1. **Regras determinísticas de redirect** (`padroes_redirect.xlsx`) — resolve casos-armadilha e de segurança crítica.
2. **Classificador BERTimbau fine-tuned** — mapeia relato → classe de falha (0-9).
3. **Base de conhecimento canônica** (`base_conhecimento_classes.xlsx`) — resposta direta para casos de alta confiança.
4. **Retrieval-Augmented Generation** sobre manuais Chevrolet (ChromaDB) — contexto adicional para casos ambíguos.

### Fluxo de decisão

```
Relato do usuário
    │
    ├─► [ETAPA 1] Regras de redirect
    │       Se alguma regra casar → resposta direta (latência ~10ms)
    │
    ├─► [ETAPA 2] BERTimbau classifica → {classe, confiança}
    │
    ├─► [ETAPA 3a] Se confiança ≥ 0.85 + keyword_alta_confianca presente:
    │       → resposta canônica da base de conhecimento (~50ms)
    │
    └─► [ETAPA 3b] Caso contrário:
            RAG nos manuais + LLM com contexto estruturado (~2-5s)
```

### Expectativa realista para o TCC

- **30-50% dos relatos** devem ser resolvidos nas etapas 1 ou 3a, sem LLM.
- Isso permite apresentar números concretos de latência média e distribuição de decisões.
- Casos que caem na etapa 3b (pipeline completo) devem ser a minoria e justificam o RAG.

---
## Seção 0 — Setup do ambiente (Colab)

Este notebook assume execução no **Google Colab** com Drive montado. Ajuste `BASE_DIR` se rodar localmente.

In [ ]:
# Montar Google Drive (pule esta célula se estiver rodando localmente)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/Agente Mecânico'
    EM_COLAB = True
except ImportError:
    BASE_DIR = './'  # ajuste conforme seu ambiente
    EM_COLAB = False

print(f'BASE_DIR: {BASE_DIR}')
print(f'Em Colab: {EM_COLAB}')

Mounted at /content/drive
BASE_DIR: /content/drive/MyDrive/Agente Mecânico
Em Colab: True


In [ ]:
# Instalação de dependências. Use --quiet para manter a saída limpa.
# O pacote 'unstructured' é pesado — só instale se for rodar a ingestão do RAG.

%pip install --quiet \
    pandas openpyxl \
    scikit-learn \
    transformers torch accelerate \
    sentence-transformers \
    chromadb \
    anthropic \
    tiktoken

# Se for rodar a ingestão dos PDFs, descomente:
# %pip install --quiet 'unstructured[pdf]' pypdf

print(' Dependências instaladas.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.9/635.9 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

In [ ]:
import os
import re
import json
import time
import hashlib
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional, Literal
from enum import Enum

import pandas as pd
import numpy as np

# ── Caminhos de entrada/saída ──────────────────────────────────────────
BASE = Path(BASE_DIR)
DATASET_PATH      = BASE / 'Dataset_Estruturado' / 'dataset_mao_na_roda_v2.xlsx'
BASE_CONHEC_PATH  = BASE / 'Dataset_Estruturado' / 'base_conhecimento_classes.xlsx'
REDIRECT_PATH     = BASE / 'Dataset_Estruturado' / 'padroes_redirect.xlsx'
MANUAIS_DIR       = BASE / 'Dataset_Nao_Estruturado'
CHROMA_DIR        = BASE / 'chromadb'
RESULTADOS_DIR    = BASE / 'resultados_pipeline'
RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)

# ── Configurações globais ──────────────────────────────────────────────
RANDOM_STATE = 42
CONFIANCA_ALTA_MIN = 0.85  # threshold para atalho da base canônica

# Nomes legíveis das classes (alinhado com o dataset v2)
CLASSES = {
    0: 'Bateria/Sistema Elétrico',
    1: 'Sistema de Freios',
    2: 'Superaquecimento do Motor',
    3: 'Suspensão/Amortecedores',
    4: 'Sistema de Transmissão/Câmbio',
    5: 'Vazamento de Óleo',
    6: 'Sistema de Arrefecimento',
    7: 'Pneu/Roda',
    8: 'Sistema de Injeção/Combustível',
    9: 'Sistema de Escapamento',
}

print(' Imports e configurações prontos.')

 Imports e configurações prontos.


---
## Seção 1 — Estruturas de dados do pipeline

Definimos os tipos que representam cada etapa do fluxo. Usar `dataclass` torna o código mais legível e facilita debug/logging.

In [ ]:
class EtapaResolucao(str, Enum):
    """Em qual etapa do pipeline o relato foi resolvido."""
    REDIRECT           = 'redirect'         # Etapa 1 — regra determinística
    CANONICA           = 'canonica'         # Etapa 3a — alta confiança + keyword
    RAG_LLM            = 'rag_llm'          # Etapa 3b — fluxo completo
    FALHA              = 'falha'            # erro no pipeline

class NivelUrgencia(str, Enum):
    BAIXA    = 'baixa'
    MEDIA    = 'média'
    ALTA     = 'alta'
    CRITICA  = 'crítica'
    NENHUMA  = 'nenhuma'

@dataclass
class ResultadoClassificador:
    classe_id: int
    classe_nome: str
    confianca: float
    probabilidades: dict  # {classe_id: prob}

@dataclass
class ChunkRecuperado:
    texto: str
    metadata: dict
    distancia: float

@dataclass
class RespostaMaoNaRoda:
    """Resposta final devolvida ao usuário + metadata interna para auditoria."""
    # Campos principais exibidos ao usuário
    texto_resposta:   str
    urgencia:         str
    exige_reboque:    bool
    classe_predita:   Optional[int]
    nome_sistema:     Optional[str]
    fontes_citadas:   list  # lista de dicts {manual, seção, modelo, ano}

    # Campos internos (para análise e TCC)
    etapa_resolucao:  EtapaResolucao
    latencia_ms:      float
    regra_acionada:   Optional[str]     = None
    confianca_bert:   Optional[float]   = None
    n_chunks_usados:  int               = 0
    tokens_llm:       Optional[int]     = None
    custo_estimado_usd: Optional[float] = None

    def resumo(self) -> str:
        """Retorna apenas a parte visível ao usuário final."""
        linhas = [
            f'**Sistema:** {self.nome_sistema or "(não identificado)"}',
            f'**Urgência:** {self.urgencia}',
        ]
        if self.exige_reboque:
            linhas.append('** Este caso recomenda REBOQUE. Não dirija até a oficina.**')
        linhas.append('')
        linhas.append(self.texto_resposta)
        if self.fontes_citadas:
            linhas.append('')
            linhas.append('**Fontes consultadas:**')
            for f in self.fontes_citadas:
                linhas.append(f'- {f.get("modelo", "?")} {f.get("ano", "?")}, seção "{f.get("secao", "?")}"')
        return '\n'.join(linhas)

print(' Estruturas de dados definidas.')

 Estruturas de dados definidas.


---
## Seção 2 — Carregamento das 3 planilhas

Validamos schema e distribuições antes de seguir.

In [ ]:
# ── Dataset principal (para referência e testes) ───────────────────────
df_dataset = pd.read_excel(DATASET_PATH)
print(f' Dataset: {len(df_dataset)} relatos, {df_dataset.shape[1]} colunas')

# ── Base de conhecimento canônica ──────────────────────────────────────
df_base = pd.read_excel(BASE_CONHEC_PATH)
assert len(df_base) == 10, f'Esperado 10 classes, encontrado {len(df_base)}'
assert set(df_base['classe_id']) == set(range(10)), 'classe_id deve cobrir 0-9'
print(f' Base canônica: {len(df_base)} classes')

# Indexar por classe_id para lookup O(1)
base_por_classe = df_base.set_index('classe_id').to_dict('index')

# ── Regras de redirect ─────────────────────────────────────────────────
df_redirect = pd.read_excel(REDIRECT_PATH)
print(f' Regras redirect: {len(df_redirect)} regras')
print(f'   Tipos de ação: {df_redirect["tipo_acao"].value_counts().to_dict()}')

# Converter NaN para string vazia nas colunas de keywords
for col in ['keywords_obrigatorias', 'keywords_contexto', 'keywords_bloqueio']:
    df_redirect[col] = df_redirect[col].fillna('')

 Dataset: 169 relatos, 15 colunas
 Base canônica: 10 classes
 Regras redirect: 10 regras
   Tipos de ação: {'escalonar_urgencia_maxima': 4, 'resposta_simples': 3, 'explicar_normalidade': 1, 'forcar_classificacao': 1, 'seguir_pipeline_rag': 1}


---
## Seção 3 — Etapa 1: Motor de Regras Determinísticas

Aplica as regras de `padroes_redirect.xlsx` na ordem em que aparecem. Primeira regra que casa vence.

**Lógica de casamento:**
- `keywords_obrigatorias`: pelo menos uma delas (separadas por `|`) deve aparecer no relato.
- `keywords_contexto`: pelo menos uma delas deve aparecer (reforça intenção).
- `keywords_bloqueio`: **nenhuma** delas pode aparecer (descarta falsos positivos).

A comparação é feita em versão normalizada do texto (lowercase, sem acentos).

In [ ]:
import unicodedata

def normalizar(texto: str) -> str:
    """Lowercase + remove acentos para comparação robusta de keywords."""
    t = texto.lower().strip()
    t = unicodedata.normalize('NFD', t)
    t = ''.join(c for c in t if unicodedata.category(c) != 'Mn')
    return t

def keyword_presente(relato_norm: str, keywords_pipe: str) -> bool:
    """Retorna True se alguma keyword (separada por |) está no relato."""
    if not keywords_pipe:
        return False
    for kw in keywords_pipe.split('|'):
        kw = normalizar(kw.strip())
        if kw and kw in relato_norm:
            return True
    return False

def avaliar_regras_redirect(relato: str, df_regras: pd.DataFrame) -> Optional[pd.Series]:
    """
    Percorre regras de cima para baixo. Retorna a primeira que casar, ou None
    se só a regra catch-all (PORTA_RAG_GENERICO) casar.
    """
    relato_norm = normalizar(relato)
    for _, regra in df_regras.iterrows():
        # Catch-all: deixa para o pipeline seguir
        if regra['tipo_acao'] == 'seguir_pipeline_rag':
            return None

        # 1. Obrigatórias: pelo menos uma precisa estar presente
        if regra['keywords_obrigatorias']:
            if not keyword_presente(relato_norm, regra['keywords_obrigatorias']):
                continue

        # 2. Contexto: pelo menos uma precisa estar presente (se especificado)
        if regra['keywords_contexto']:
            if not keyword_presente(relato_norm, regra['keywords_contexto']):
                continue

        # 3. Bloqueio: NENHUMA pode estar presente
        if regra['keywords_bloqueio']:
            if keyword_presente(relato_norm, regra['keywords_bloqueio']):
                continue

        # Todas as condições atendidas
        return regra
    return None

# ── Teste rápido ───────────────────────────────────────────────────────
relatos_teste = [
    ('saiu fumaça branca no cano de escape só no frio, some depois', 'COND_NORMAL_01'),
    ('junta do cabeçote furou, óleo misturando com água',            'JUNTA_CABECOTE_01'),
    ('pedal do freio afunda todo e não freia',                       'FREIO_CRITICO_01'),
    ('deixei a luz interna acesa a noite toda e agora não pega',     'BATERIA_SIMPLES_01'),
    ('carro com barulho estranho no motor',                          None),  # não deve casar
]

for relato, esperado in relatos_teste:
    regra = avaliar_regras_redirect(relato, df_redirect)
    obtido = regra['padrao_id'] if regra is not None else None
    status = '✓' if obtido == esperado else '✗'
    print(f'{status} "{relato[:50]}..."  →  {obtido} (esperado: {esperado})')

✓ "saiu fumaça branca no cano de escape só no frio, s..."  →  COND_NORMAL_01 (esperado: COND_NORMAL_01)
✓ "junta do cabeçote furou, óleo misturando com água..."  →  JUNTA_CABECOTE_01 (esperado: JUNTA_CABECOTE_01)
✓ "pedal do freio afunda todo e não freia..."  →  FREIO_CRITICO_01 (esperado: FREIO_CRITICO_01)
✗ "deixei a luz interna acesa a noite toda e agora nã..."  →  PRESSAO_OLEO_01 (esperado: BATERIA_SIMPLES_01)
✓ "carro com barulho estranho no motor..."  →  None (esperado: None)


---
## Seção 4 — Etapa 2: Classificador (BERTimbau ou Baseline)

Carregamos o modelo treinado se disponível, ou caímos no **baseline TF-IDF + Logistic Regression** como fallback.

Essa flexibilidade é importante: o notebook deve rodar **antes** do fine-tuning do BERTimbau estar pronto. Com o baseline, você já consegue testar o pipeline completo e comparar métricas depois.

In [ ]:
# ── Tentar carregar BERTimbau fine-tuned ────────────────────────────────
MODELO_BERT_PATH = BASE / 'modelos' / 'bertimbau_mao_na_roda'
bert_disponivel = MODELO_BERT_PATH.exists() and (MODELO_BERT_PATH / 'config.json').exists()

if bert_disponivel:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    import torch
    print(' BERTimbau fine-tuned encontrado. Carregando...')
    tokenizer_bert = AutoTokenizer.from_pretrained(str(MODELO_BERT_PATH))
    model_bert = AutoModelForSequenceClassification.from_pretrained(str(MODELO_BERT_PATH))
    model_bert.eval()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model_bert.to(device)
    print(f'   Device: {device}')
else:
    print(' BERTimbau fine-tuned não encontrado. Usando BASELINE TF-IDF+LR.')
    print(f'   (esperado em: {MODELO_BERT_PATH})')

# ── Baseline sempre disponível como fallback ───────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Pré-processamento simples (alinhado com o notebook baseline)
# ═══════════════════════════════════════════════════════════════════════
# TODO — SUBSTITUIR POR PIPELINE V3 ADAPTADO (Honorato)
# ═══════════════════════════════════════════════════════════════════════
# A função `preprocessar()` abaixo é uma versão mínima para bootstrapping.
# Substituir pela adaptação do pipeline_v3.py (pipelinev3.py de João Pedro
# Honorato, laboratório CIBERDEM/Mackenzie) com as seguintes configurações:
#
#   preprocess(texto,
#       substituir_users=False,      # não há @usuarios em relatos de carro
#       substituir_emojis=False,     # idem
#       remover_urls=True,           # segurança (usuário pode colar link)
#       normalizar=True,             # Enelvo corrige "vc", "pq", "tá", etc.
#       converter_ascii=True,
#       remover_pontuacao=True,
#       tokenizar_texto=False,       # manter como string para BERTimbau/TF-IDF
#   )
#
# + estender `dicionario_girias_completo` com vocabulário automotivo:
#     "ronca o motor": "motor com ruído forte",
#     "engasga": "falha intermitente",
#     "tá pifando": "apresentando defeito",
#     "bate pino": "ruído metálico no motor",
#     "patina": "embreagem com falha",
#     "tá ferrando": "apresentando defeito",
#     "mó barulho": "barulho forte",
#     "mó ronco": "ruído intenso",
#     ... (ampliar conforme expansão do dataset)
#
# Dependências: pip install enelvo demoji unidecode nltk
# Referência: pipeline usado no TCC de João Pedro Honorato.
# ═══════════════════════════════════════════════════════════════════════

STOPWORDS_PT = {
    'o','a','os','as','e','de','do','da','dos','das','em','no','na','nos','nas',
    'um','uma','que','se','por','com','para','ao','aos','à','às','ou','mas','mais',
    'não','nao','é','meu','minha','quando','depois','antes','muito','bem','só','lá',
    'pra','pro','isso','esse','essa','este','esta','eu','você','vc','ta','tá','to','tô',
}

def preprocessar(texto: str) -> str:
    t = str(texto).lower().strip()
    t = re.sub(r'[^a-záéíóúãõâêôàüç\s]', ' ', t)
    tokens = [w for w in t.split() if w not in STOPWORDS_PT and len(w) > 1]
    return ' '.join(tokens)

# Treinar baseline se BERTimbau não está disponível
if not bert_disponivel:
    print('\n Treinando baseline no dataset_v2...')
    df_train = df_dataset.copy()
    df_train['relato_processado'] = df_train['relato'].apply(preprocessar)
    X_train = df_train['relato_processado'].values
    y_train = df_train['falha_label'].values

    pipeline_baseline = Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=600, ngram_range=(1, 2),
            strip_accents='unicode', sublinear_tf=True, min_df=1
        )),
        ('clf', LogisticRegression(
            C=2.0, solver='lbfgs', max_iter=1000,
            random_state=RANDOM_STATE, class_weight='balanced'
        )),
    ])
    pipeline_baseline.fit(X_train, y_train)
    print(f'   Treinado em {len(X_train)} relatos.')

 BERTimbau fine-tuned não encontrado. Usando BASELINE TF-IDF+LR.
   (esperado em: /content/drive/MyDrive/Agente Mecânico/modelos/bertimbau_mao_na_roda)

 Treinando baseline no dataset_v2...
   Treinado em 169 relatos.


In [ ]:
def classificar(relato: str) -> ResultadoClassificador:
    """
    Retorna classe predita + confiança. Usa BERTimbau se disponível,
    baseline caso contrário. Interface única, chamador não precisa saber.
    """
    if bert_disponivel:
        import torch
        inputs = tokenizer_bert(
            relato, return_tensors='pt',
            truncation=True, max_length=128, padding=True
        ).to(device)
        with torch.no_grad():
            logits = model_bert(**inputs).logits
            probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    else:
        relato_proc = preprocessar(relato)
        probs = pipeline_baseline.predict_proba([relato_proc])[0]

    classe_id = int(np.argmax(probs))
    return ResultadoClassificador(
        classe_id=classe_id,
        classe_nome=CLASSES[classe_id],
        confianca=float(probs[classe_id]),
        probabilidades={i: float(p) for i, p in enumerate(probs)},
    )

# Teste
for rel in ['carro não dá partida de manhã',
            'ponteiro da temperatura subiu todo',
            'freio está fazendo barulho ao pisar']:
    r = classificar(rel)
    print(f'"{rel}"')
    print(f'   → Classe {r.classe_id}: {r.classe_nome} (conf={r.confianca:.2%})')

"carro não dá partida de manhã"
   → Classe 3: Suspensão/Amortecedores (conf=19.69%)
"ponteiro da temperatura subiu todo"
   → Classe 2: Superaquecimento do Motor (conf=26.70%)
"freio está fazendo barulho ao pisar"
   → Classe 1: Sistema de Freios (conf=47.82%)


---
## Seção 5 — Etapa 3a: Resposta Canônica

Se a confiança do classificador é alta **e** há uma keyword forte daquela classe no relato, devolvemos a resposta já escrita na base de conhecimento. Essa etapa **não chama LLM**.

In [ ]:
def tem_keyword_alta_confianca(relato: str, classe_id: int) -> bool:
    """Verifica se alguma keyword de alta confiança daquela classe está no relato."""
    kws = base_por_classe[classe_id].get('keywords_alta_confianca', '')
    if pd.isna(kws) or not kws:
        return False
    relato_norm = normalizar(relato)
    for kw in kws.split(','):
        if keyword_presente(relato_norm, kw.strip()):
            return True
    return False

def gerar_resposta_canonica(relato: str, classif: ResultadoClassificador,
                             latencia_ms: float) -> RespostaMaoNaRoda:
    """Monta resposta diretamente da base de conhecimento (sem LLM)."""
    info = base_por_classe[classif.classe_id]

    # Determinar urgência via regra: se sintoma crítico descrito, eleva
    urgencia_base = 'média'
    sinais_crit = str(info.get('sinais_urgencia_critica', ''))
    if any(keyword_presente(normalizar(relato), normalizar(s))
           for s in sinais_crit.split('.') if len(s.strip()) > 15):
        urgencia_base = 'crítica'

    return RespostaMaoNaRoda(
        texto_resposta=str(info['frase_resposta_canonica']),
        urgencia=urgencia_base,
        exige_reboque=bool(info.get('exige_reboque', False)),
        classe_predita=classif.classe_id,
        nome_sistema=str(info['nome_sistema']),
        fontes_citadas=[],
        etapa_resolucao=EtapaResolucao.CANONICA,
        latencia_ms=latencia_ms,
        confianca_bert=classif.confianca,
        n_chunks_usados=0,
        tokens_llm=0,
        custo_estimado_usd=0.0,
    )

print(' Função de resposta canônica pronta.')

 Função de resposta canônica pronta.


---
## Seção 6 — Etapa 3b: RAG sobre manuais Chevrolet

Se o relato cai aqui (confiança baixa ou sem keyword forte), invocamos o pipeline completo:
1. Busca vetorial no ChromaDB filtrada por `tipo_conteudo` relevante e classe predita.
2. Monta prompt estruturado com contexto dos chunks + linha da base canônica.
3. Chama LLM (Claude ou alternativa) para sintetizar resposta personalizada.

> **Pré-requisito:** o ChromaDB já deve estar povoado. Se não estiver, a célula de setup abaixo detecta e alerta.

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

# ── Conectar (ou criar) coleção ────────────────────────────────────────
client_chroma = chromadb.PersistentClient(path=str(CHROMA_DIR))

try:
    colecao_rag = client_chroma.get_collection(name='mao_na_roda_manuais')
    n_chunks = colecao_rag.count()
    rag_disponivel = n_chunks > 0
    print(f' ChromaDB: coleção "mao_na_roda_manuais" com {n_chunks} chunks')
except Exception as e:
    rag_disponivel = False
    colecao_rag = None
    print(f' ChromaDB: coleção não encontrada ou vazia.')
    print(f'   → rode primeiro o notebook de ingestão dos PDFs.')
    print(f'   → detalhe: {e}')

# ── Modelo de embeddings (precisa ser o mesmo da ingestão) ─────────────
if rag_disponivel:
    print(' Carregando modelo de embeddings...')
    emb_model = SentenceTransformer('BAAI/bge-m3')
    print('   Modelo: BAAI/bge-m3 (multilingual)')
else:
    emb_model = None

 ChromaDB: coleção não encontrada ou vazia.
   → rode primeiro o notebook de ingestão dos PDFs.
   → detalhe: Collection [mao_na_roda_manuais] does not exist


In [ ]:
def recuperar_chunks(relato: str, classe_predita: int, top_k: int = 3,
                      veiculo: Optional[dict] = None) -> list:
    """Retorna os chunks mais relevantes dos manuais, filtrados por contexto."""
    if not rag_disponivel:
        return []

    # Embedding da query
    query_embedding = emb_model.encode(relato, normalize_embeddings=True).tolist()

    # Tipos de conteúdo úteis para diagnóstico (excluímos operação normal, etc.)
    tipos_relevantes = [
        'luz_painel', 'procedimento_emergencia', 'procedimento_manutencao',
        'advertencia_seguranca', 'descricao_componente', 'sintoma_causa',
    ]

    # Filtro de metadata
    filtros = {
        '$and': [
            {'tipo_conteudo': {'$in': tipos_relevantes}},
            {'tem_sistemas':  {'$eq': True}},
        ]
    }

    # Filtro opcional por veículo
    if veiculo:
        if veiculo.get('modelo'):
            filtros['$and'].append({'modelo': {'$eq': veiculo['modelo']}})
        if veiculo.get('ano'):
            ano = int(veiculo['ano'])
            filtros['$and'].append({'ano_modelo': {'$gte': ano - 3}})
            filtros['$and'].append({'ano_modelo': {'$lte': ano + 3}})

    # Over-fetch para depois filtrar por classe em Python
    resultados = colecao_rag.query(
        query_embeddings=[query_embedding],
        n_results=top_k * 4,
        where=filtros,
    )

    # Pós-filtro: classe_predita deve estar em sistemas_mapeados do chunk
    chunks_filtrados = []
    for doc, meta, dist in zip(
        resultados['documents'][0],
        resultados['metadatas'][0],
        resultados['distances'][0],
    ):
        # sistemas_mapeados está como string separada por vírgula no Chroma
        sistemas_str = str(meta.get('sistemas_mapeados', ''))
        sistemas = [int(s) for s in sistemas_str.split(',') if s.strip().isdigit()]
        if classe_predita in sistemas:
            chunks_filtrados.append(ChunkRecuperado(
                texto=doc, metadata=meta, distancia=float(dist)
            ))
        if len(chunks_filtrados) >= top_k:
            break

    # Se o filtro estrito esvaziou, relaxar (pegar top-k sem filtro de classe)
    if not chunks_filtrados:
        for doc, meta, dist in zip(
            resultados['documents'][0][:top_k],
            resultados['metadatas'][0][:top_k],
            resultados['distances'][0][:top_k],
        ):
            chunks_filtrados.append(ChunkRecuperado(
                texto=doc, metadata=meta, distancia=float(dist)
            ))

    return chunks_filtrados

print(' Função de recuperação de contexto pronta.')

 Função de recuperação de contexto pronta.


In [ ]:
# ── Cliente LLM (Anthropic Claude) ─────────────────────────────────────
# Substitua a ANTHROPIC_API_KEY pelos seus dados (use Secrets do Colab)

try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')

llm_disponivel = bool(ANTHROPIC_API_KEY)

if llm_disponivel:
    from anthropic import Anthropic
    cliente_llm = Anthropic(api_key=ANTHROPIC_API_KEY)
    MODELO_LLM = 'claude-haiku-4-5'  # leve e barato para MVP
    print(f' LLM disponível: {MODELO_LLM}')
else:
    cliente_llm = None
    print(' LLM não configurado. O pipeline 3b vai cair em fallback offline.')

# ── Template de prompt ────────────────────────────────────────────────
PROMPT_TEMPLATE = '''Você é o assistente Mão na Roda, especialista em diagnóstico automotivo preliminar.
Use APENAS as informações abaixo para responder. Se não houver informação suficiente,
seja honesto sobre a limitação e recomende procurar um mecânico.

=== RELATO DO MOTORISTA ===
"{relato}"

=== CLASSIFICAÇÃO AUTOMÁTICA ===
Sistema provavelmente afetado: {classe_nome}
Confiança da classificação: {confianca:.1%}

=== INFORMAÇÕES CANÔNICAS SOBRE ESTE SISTEMA ===
Sintomas típicos: {sintomas}
Causas prováveis: {causas}
Ações imediatas seguras: {acoes}
Sinais de urgência crítica: {sinais_criticos}
O que NÃO fazer: {nao_fazer}

=== TRECHOS DOS MANUAIS DO VEÍCULO ===
{trechos_rag}

=== INSTRUÇÕES DE RESPOSTA ===
Estruture a resposta em seções curtas:
1. Provável diagnóstico (2-3 linhas em linguagem simples)
2. Urgência (baixa/média/alta/crítica) e por quê
3. O que fazer AGORA (3-5 ações)
4. O que NÃO fazer
5. Quando procurar mecânico

Regras:
- Linguagem acessível, sem jargão técnico desnecessário
- Cite a fonte quando usar informação do manual (modelo/ano/seção)
- NUNCA garanta diagnóstico — use "provavelmente", "indica", "sugere"
- Se for caso de segurança (freio, superaquecimento, luz de óleo), priorize
  instrução de parar o veículo
- Encerre sempre com: "Este diagnóstico é preliminar e não substitui avaliação
  profissional. Em dúvida, procure um mecânico."
'''

def formatar_trechos_rag(chunks: list) -> str:
    """Converte lista de ChunkRecuperado em texto estruturado para o prompt."""
    if not chunks:
        return '(nenhum trecho relevante encontrado nos manuais)'
    partes = []
    for i, c in enumerate(chunks, 1):
        modelo = c.metadata.get('modelo', '?')
        ano = c.metadata.get('ano_modelo', '?')
        titulo = c.metadata.get('titulo_chunk', 'sem título')
        partes.append(f'[Fonte {i}: Chevrolet {modelo} {ano} — {titulo}]\n{c.texto}\n')
    return '\n'.join(partes)

 LLM não configurado. O pipeline 3b vai cair em fallback offline.


In [ ]:
# Custos aproximados de Claude Haiku 4.5 em 2026 (verificar na doc oficial)
CUSTO_INPUT_POR_1M  = 1.00   # USD por 1M tokens de input
CUSTO_OUTPUT_POR_1M = 5.00   # USD por 1M tokens de output

def gerar_resposta_llm(relato: str, classif: ResultadoClassificador,
                        chunks: list, latencia_ms_parcial: float) -> RespostaMaoNaRoda:
    """Monta prompt e chama LLM. Se LLM indisponível, devolve fallback offline."""
    info = base_por_classe[classif.classe_id]

    # Fallback quando LLM não está disponível
    if not llm_disponivel:
        texto = (
            f'[Modo offline — LLM não configurado]\n\n'
            f'Sistema provavelmente afetado: {classif.classe_nome} '
            f'(confiança: {classif.confianca:.0%})\n\n'
            f'{info["frase_resposta_canonica"]}\n\n'
            f'Consulte os trechos dos manuais recuperados para mais detalhes.'
        )
        return RespostaMaoNaRoda(
            texto_resposta=texto,
            urgencia='média',
            exige_reboque=bool(info.get('exige_reboque', False)),
            classe_predita=classif.classe_id,
            nome_sistema=str(info['nome_sistema']),
            fontes_citadas=[
                {'modelo': c.metadata.get('modelo'), 'ano': c.metadata.get('ano_modelo'),
                 'secao': c.metadata.get('titulo_chunk')}
                for c in chunks
            ],
            etapa_resolucao=EtapaResolucao.RAG_LLM,
            latencia_ms=latencia_ms_parcial,
            confianca_bert=classif.confianca,
            n_chunks_usados=len(chunks),
            tokens_llm=0,
            custo_estimado_usd=0.0,
        )

    # Montar prompt
    prompt = PROMPT_TEMPLATE.format(
        relato=relato,
        classe_nome=classif.classe_nome,
        confianca=classif.confianca,
        sintomas=info.get('sintomas_tipicos', ''),
        causas=info.get('causas_provaveis_ordenadas', ''),
        acoes=info.get('acoes_imediatas_seguras', ''),
        sinais_criticos=info.get('sinais_urgencia_critica', ''),
        nao_fazer=info.get('o_que_nao_fazer', ''),
        trechos_rag=formatar_trechos_rag(chunks),
    )

    # Chamar LLM
    t0 = time.time()
    resp = cliente_llm.messages.create(
        model=MODELO_LLM,
        max_tokens=700,
        messages=[{'role': 'user', 'content': prompt}],
    )
    dt_llm = (time.time() - t0) * 1000

    texto_final = resp.content[0].text
    tokens_in  = resp.usage.input_tokens
    tokens_out = resp.usage.output_tokens
    custo = (tokens_in * CUSTO_INPUT_POR_1M + tokens_out * CUSTO_OUTPUT_POR_1M) / 1_000_000

    # Extrair urgência da resposta (heurística simples)
    urg_match = re.search(r'urg[êe]ncia[^a-z]*?(cr[íi]tica|alta|m[ée]dia|baixa)',
                           texto_final.lower())
    urgencia = urg_match.group(1) if urg_match else 'média'
    # Normalizar com acento
    mapa_urg = {'critica': 'crítica', 'media': 'média', 'alta': 'alta', 'baixa': 'baixa'}
    urgencia = mapa_urg.get(urgencia, urgencia)

    return RespostaMaoNaRoda(
        texto_resposta=texto_final,
        urgencia=urgencia,
        exige_reboque=bool(info.get('exige_reboque', False)),
        classe_predita=classif.classe_id,
        nome_sistema=str(info['nome_sistema']),
        fontes_citadas=[
            {'modelo': c.metadata.get('modelo'), 'ano': c.metadata.get('ano_modelo'),
             'secao': c.metadata.get('titulo_chunk')}
            for c in chunks
        ],
        etapa_resolucao=EtapaResolucao.RAG_LLM,
        latencia_ms=latencia_ms_parcial + dt_llm,
        confianca_bert=classif.confianca,
        n_chunks_usados=len(chunks),
        tokens_llm=tokens_in + tokens_out,
        custo_estimado_usd=custo,
    )

print(' Função de geração via LLM pronta.')

 Função de geração via LLM pronta.


---
## Seção 7 — Função Orquestradora

Esta é a **função principal do agente**. Executa o fluxo de 3 etapas e devolve uma `RespostaMaoNaRoda` completa.

In [ ]:
def resolver_relato(relato: str, veiculo: Optional[dict] = None,
                     verbose: bool = False) -> RespostaMaoNaRoda:
    """
    Função principal do agente Mão na Roda.

    Args:
        relato: texto do sintoma descrito pelo motorista
        veiculo: {'modelo': 'corsa', 'ano': 2011} para filtrar RAG (opcional)
        verbose: se True, imprime passos intermediários

    Returns:
        RespostaMaoNaRoda com texto, metadata e rastreamento de etapas.
    """
    t_inicio = time.time()

    # ═══ ETAPA 1: Regras de redirect ════════════════════════════════════
    if verbose: print(f'[1] Avaliando regras de redirect...')
    regra = avaliar_regras_redirect(relato, df_redirect)

    if regra is not None:
        if verbose: print(f'    ✓ Regra casou: {regra["padrao_id"]}')
        latencia = (time.time() - t_inicio) * 1000

        classe_id = regra.get('classe_sugerida')
        # Se classe_sugerida é NaN (pandas), trata como None
        if pd.isna(classe_id):
            classe_id = None
        else:
            classe_id = int(classe_id)

        nome_sistema = CLASSES.get(classe_id) if classe_id is not None else None
        exige_reboque = False
        if classe_id is not None:
            exige_reboque = bool(base_por_classe[classe_id].get('exige_reboque', False))
        # Regras de urgência crítica sempre exigem reboque
        if regra['tipo_acao'] == 'escalonar_urgencia_maxima':
            exige_reboque = True

        return RespostaMaoNaRoda(
            texto_resposta=str(regra['resposta_direta']),
            urgencia=str(regra['urgencia_real']),
            exige_reboque=exige_reboque,
            classe_predita=classe_id,
            nome_sistema=nome_sistema,
            fontes_citadas=[],
            etapa_resolucao=EtapaResolucao.REDIRECT,
            latencia_ms=latencia,
            regra_acionada=str(regra['padrao_id']),
            confianca_bert=None,
            n_chunks_usados=0,
            tokens_llm=0,
            custo_estimado_usd=0.0,
        )

    # ═══ ETAPA 2: Classificador ═════════════════════════════════════════
    if verbose: print(f'[2] Classificando...')
    classif = classificar(relato)
    if verbose: print(f'    → Classe {classif.classe_id} ({classif.classe_nome}) '
                      f'| confiança {classif.confianca:.2%}')

    # ═══ ETAPA 3a: Resposta canônica (se alta confiança + keyword) ═════
    tem_kw = tem_keyword_alta_confianca(relato, classif.classe_id)
    if classif.confianca >= CONFIANCA_ALTA_MIN and tem_kw:
        if verbose: print(f'[3a] Alta confiança + keyword detectada → resposta canônica')
        latencia = (time.time() - t_inicio) * 1000
        return gerar_resposta_canonica(relato, classif, latencia)

    # ═══ ETAPA 3b: RAG + LLM ════════════════════════════════════════════
    if verbose: print(f'[3b] Pipeline completo: RAG + LLM')
    chunks = recuperar_chunks(relato, classif.classe_id, top_k=3, veiculo=veiculo)
    if verbose: print(f'    → {len(chunks)} chunks recuperados')
    latencia_ate_agora = (time.time() - t_inicio) * 1000
    return gerar_resposta_llm(relato, classif, chunks, latencia_ate_agora)

print(' Orquestrador pronto.')

 Orquestrador pronto.


---
## Seção 8 — Smoke test com 5 relatos representativos

Testamos o pipeline completo em exemplos que cobrem as 3 etapas, para confirmar que cada caminho funciona.

In [ ]:
relatos_smoke = [
    # Etapa 1 — regra de redirect (segurança crítica)
    ('pedal do freio afunda todo e não freia nada', 'ETAPA 1 esperada'),
    # Etapa 1 — condição normal
    ('sai fumaça branca no cano só no frio, some depois', 'ETAPA 1 esperada'),
    # Etapa 3a — alta confiança + keyword
    ('carro não liga, só dá um clique quando giro a chave', 'ETAPA 3a esperada'),
    # Etapa 3b — caso ambíguo
    ('sinto um barulho diferente vindo da frente do carro', 'ETAPA 3b esperada'),
    # Etapa 1 — fluido de transmissão queimado (linguagem gíria)
    ('ponteiro da temperatura subiu tudo, tá no vermelho', 'ETAPA 1 esperada'),
]

print('=' * 75)
print(f'SMOKE TEST — {len(relatos_smoke)} relatos')
print('=' * 75)

for i, (relato, esperado) in enumerate(relatos_smoke, 1):
    print(f'\n{"─" * 75}')
    print(f'[{i}/{len(relatos_smoke)}] "{relato}"')
    print(f'       ({esperado})')
    print('─' * 75)

    resposta = resolver_relato(relato, verbose=True)

    print(f'\n>>> RESULTADO:')
    print(f'    Etapa:      {resposta.etapa_resolucao.value}')
    print(f'    Urgência:   {resposta.urgencia}')
    print(f'    Reboque:    {"SIM" if resposta.exige_reboque else "não"}')
    print(f'    Latência:   {resposta.latencia_ms:.0f}ms')
    if resposta.tokens_llm:
        print(f'    Tokens LLM: {resposta.tokens_llm}')
        print(f'    Custo:      US${resposta.custo_estimado_usd:.5f}')
    print(f'\n{resposta.texto_resposta[:400]}...' if len(resposta.texto_resposta) > 400
          else f'\n{resposta.texto_resposta}')

SMOKE TEST — 5 relatos

───────────────────────────────────────────────────────────────────────────
[1/5] "pedal do freio afunda todo e não freia nada"
       (ETAPA 1 esperada)
───────────────────────────────────────────────────────────────────────────
[1] Avaliando regras de redirect...
    ✓ Regra casou: FREIO_CRITICO_01

>>> RESULTADO:
    Etapa:      redirect
    Urgência:   crítica
    Reboque:    SIM
    Latência:   1ms

EMERGÊNCIA DE SEGURANÇA. Pedal de freio afundando indica perda hidráulica. NÃO DIRIJA. Acione o freio de mão gradualmente (não de uma vez, para não travar) e pare em local seguro. Chame guincho. Conduzir nessas condições é risco de morte. Causa comum: vazamento no sistema ou ar entrando. Nunca complete o fluido com tipo errado — veja no manual (geralmente DOT 3 ou DOT 4).

───────────────────────────────────────────────────────────────────────────
[2/5] "sai fumaça branca no cano só no frio, some depois"
       (ETAPA 1 esperada)
────────────────────────────────

---
## Seção 9 — Avaliação em lote no conjunto de testes

Rodamos o pipeline em um hold-out estratificado do dataset e coletamos métricas agregadas:

- **Distribuição de etapas** — quantos % dos relatos resolveram em cada etapa
- **Latência média por etapa**
- **Custo total estimado em USD**
- **Acurácia do classificador** (quando disponível no ground truth)

Estes são os números que vão compor a discussão de resultados do TCC.

In [ ]:
from sklearn.model_selection import train_test_split

# Amostra estratificada por classe (15% do dataset)
_, df_teste = train_test_split(
    df_dataset, test_size=0.15,
    stratify=df_dataset['falha_label'],
    random_state=RANDOM_STATE,
)
df_teste = df_teste.reset_index(drop=True)
print(f' Conjunto de teste: {len(df_teste)} relatos')
print(f'   Distribuição: {dict(df_teste["falha_label"].value_counts().sort_index())}')

# Executar pipeline em todos
print('\n Rodando pipeline no conjunto de teste...')
resultados_lote = []
for i, row in df_teste.iterrows():
    resp = resolver_relato(row['relato'], verbose=False)
    resultados_lote.append({
        'id':              int(row['id']),
        'relato':          row['relato'],
        'classe_verdadeira': int(row['falha_label']),
        'classe_predita':  resp.classe_predita,
        'urgencia':        resp.urgencia,
        'etapa':           resp.etapa_resolucao.value,
        'regra':           resp.regra_acionada,
        'latencia_ms':     round(resp.latencia_ms, 1),
        'confianca':       resp.confianca_bert,
        'n_chunks':        resp.n_chunks_usados,
        'tokens':          resp.tokens_llm,
        'custo_usd':       resp.custo_estimado_usd or 0.0,
    })
    if (i + 1) % 5 == 0:
        print(f'   {i+1}/{len(df_teste)}')

df_res = pd.DataFrame(resultados_lote)
df_res.to_csv(RESULTADOS_DIR / 'avaliacao_pipeline.csv', index=False)
print(f'\n Resultados salvos em: {RESULTADOS_DIR}/avaliacao_pipeline.csv')

In [ ]:
# ── Distribuição de etapas de resolução ────────────────────────────────
print('=' * 60)
print('DISTRIBUIÇÃO DE ETAPAS DE RESOLUÇÃO')
print('=' * 60)
dist_etapas = df_res['etapa'].value_counts(normalize=True) * 100
for etapa, pct in dist_etapas.items():
    n = int(df_res['etapa'].value_counts()[etapa])
    print(f'  {etapa:15s}: {pct:5.1f}% ({n} relatos)')

# ── Latência por etapa ──────────────────────────────────────────────────
print('\n' + '=' * 60)
print('LATÊNCIA POR ETAPA (ms)')
print('=' * 60)
lat = df_res.groupby('etapa')['latencia_ms'].agg(['mean', 'median', 'min', 'max', 'count'])
print(lat.round(1))

# ── Custo agregado ──────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('CUSTO ESTIMADO')
print('=' * 60)
custo_total = df_res['custo_usd'].sum()
tokens_total = df_res['tokens'].sum()
print(f'  Total de tokens LLM:     {tokens_total:,}')
print(f'  Custo total USD:         ${custo_total:.4f}')
print(f'  Custo médio por relato:  ${custo_total/len(df_res):.5f}')
if tokens_total > 0:
    relatos_com_llm = (df_res['tokens'] > 0).sum()
    print(f'  Custo médio quando usa LLM: ${custo_total/relatos_com_llm:.5f}')

# ── Acurácia do classificador (só nos relatos que chegaram à etapa 2+) ─
print('\n' + '=' * 60)
print('ACURÁCIA DO CLASSIFICADOR')
print('=' * 60)
df_classif = df_res[df_res['etapa'] != 'redirect'].copy()
if len(df_classif) > 0:
    acertos = (df_classif['classe_predita'] == df_classif['classe_verdadeira']).sum()
    acc = acertos / len(df_classif)
    print(f'  Nos {len(df_classif)} relatos que passaram pelo classificador:')
    print(f'  Acertos: {acertos}/{len(df_classif)} ({acc:.1%})')
else:
    print('  Todos foram resolvidos por regras. Classificador não foi exercitado.')

# ── Urgência detectada ──────────────────────────────────────────────────
print('\n' + '=' * 60)
print('DISTRIBUIÇÃO DE URGÊNCIA NAS RESPOSTAS')
print('=' * 60)
print(df_res['urgencia'].value_counts())

In [ ]:
# ── Gráficos para o relatório ───────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Gráfico 1: distribuição de etapas
ax = axes[0]
dist_etapas_abs = df_res['etapa'].value_counts()
cores_etapa = {'redirect': '#27AE60', 'canonica': '#3498DB', 'rag_llm': '#E67E22'}
cores_plot = [cores_etapa.get(e, '#95A5A6') for e in dist_etapas_abs.index]
bars = ax.bar(dist_etapas_abs.index, dist_etapas_abs.values, color=cores_plot, edgecolor='white')
for bar, v in zip(bars, dist_etapas_abs.values):
    pct = v / len(df_res) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{v}\n({pct:.0f}%)', ha='center', fontweight='bold')
ax.set_title('Resolução por Etapa do Pipeline', fontsize=13, fontweight='bold')
ax.set_ylabel('Número de relatos')
ax.grid(axis='y', alpha=0.3)

# Gráfico 2: latência (boxplot)
ax = axes[1]
etapas_ordem = ['redirect', 'canonica', 'rag_llm']
dados_lat = [df_res[df_res['etapa'] == e]['latencia_ms'].values for e in etapas_ordem]
dados_lat = [d for d in dados_lat if len(d) > 0]
labels_validos = [e for e, d in zip(etapas_ordem, dados_lat) if len(d) > 0]
bp = ax.boxplot(dados_lat, labels=labels_validos, patch_artist=True)
for patch, etapa in zip(bp['boxes'], labels_validos):
    patch.set_facecolor(cores_etapa.get(etapa, '#95A5A6'))
    patch.set_alpha(0.7)
ax.set_title('Latência por Etapa (ms, escala log)', fontsize=13, fontweight='bold')
ax.set_ylabel('Latência (ms)')
ax.set_yscale('log')
ax.grid(axis='y', alpha=0.3)

# Gráfico 3: custo acumulado
ax = axes[2]
custo_acumulado = df_res.sort_values('id')['custo_usd'].cumsum()
ax.plot(range(1, len(custo_acumulado) + 1), custo_acumulado.values,
        color='#8E44AD', linewidth=2)
ax.fill_between(range(1, len(custo_acumulado) + 1), 0, custo_acumulado.values,
                alpha=0.3, color='#8E44AD')
ax.set_title('Custo Acumulado em LLM (USD)', fontsize=13, fontweight='bold')
ax.set_xlabel('Nº do relato processado')
ax.set_ylabel('Custo acumulado (USD)')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTADOS_DIR / 'fig_pipeline_integrado.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n Figura salva em: {RESULTADOS_DIR}/fig_pipeline_integrado.png')

---
## Seção 10 — Análise de erros e casos-fronteira

Inspecionamos manualmente os relatos em que o classificador errou, para entender padrões e informar trabalho futuro.

In [ ]:
# Erros do classificador (apenas em relatos que passaram pela etapa 2+)
df_erros = df_res[
    (df_res['etapa'] != 'redirect') &
    (df_res['classe_predita'] != df_res['classe_verdadeira'])
].copy()

if len(df_erros) > 0:
    df_erros['classe_verdadeira_nome'] = df_erros['classe_verdadeira'].map(CLASSES)
    df_erros['classe_predita_nome']  = df_erros['classe_predita'].map(CLASSES)

    print(f' Total de erros: {len(df_erros)}\n')
    for _, row in df_erros.iterrows():
        print(f'ID {row["id"]}: "{row["relato"][:80]}..."')
        print(f'  Verdadeira: {row["classe_verdadeira_nome"]}')
        print(f'  Predita:    {row["classe_predita_nome"]} (conf={row["confianca"]:.1%})')
        print()

    # Matriz de confusão dos erros
    print('Pares de confusão mais comuns:')
    pares = df_erros.groupby(['classe_verdadeira_nome', 'classe_predita_nome']).size()
    pares = pares.sort_values(ascending=False)
    for (verd, pred), count in pares.head(5).items():
        print(f'  {verd:35s} → {pred:35s}: {count}x')
else:
    print(' Nenhum erro no conjunto de teste. (pode indicar overfitting ou teste pequeno demais)')

---
## Seção 11 — Export final para o TCC

Salva um sumário estruturado com todos os números que vão compor a seção de resultados do relatório.

In [ ]:
sumario = {
    'dataset_teste': {
        'n_relatos':          len(df_res),
        'distribuicao_classes': dict(
            df_teste['falha_label'].value_counts().sort_index().to_dict()
        ),
    },
    'distribuicao_etapas_pct': {
        k: round(v, 1) for k, v in dist_etapas.to_dict().items()
    },
    'latencia_ms': {
        'media_geral':   round(float(df_res['latencia_ms'].mean()), 1),
        'mediana_geral': round(float(df_res['latencia_ms'].median()), 1),
        'por_etapa': {
            etapa: {
                'media':  round(float(df_res[df_res['etapa'] == etapa]['latencia_ms'].mean()), 1),
                'mediana': round(float(df_res[df_res['etapa'] == etapa]['latencia_ms'].median()), 1),
                'n':      int((df_res['etapa'] == etapa).sum()),
            }
            for etapa in df_res['etapa'].unique()
        },
    },
    'custo_usd': {
        'total':            round(custo_total, 4),
        'medio_por_relato': round(custo_total / len(df_res), 5),
        'tokens_total':     int(tokens_total),
    },
    'classificador': {
        'modelo_usado':     'BERTimbau' if bert_disponivel else 'TF-IDF+LR baseline',
        'n_relatos_classificados': int(len(df_classif)) if len(df_classif) > 0 else 0,
        'acuracia':         round(acc, 4) if len(df_classif) > 0 else None,
        'n_erros':          int(len(df_erros)),
    },
    'rag': {
        'disponivel':       rag_disponivel,
        'chunks_indexados': n_chunks if rag_disponivel else 0,
    },
    'configuracao': {
        'threshold_confianca_alta': CONFIANCA_ALTA_MIN,
        'n_regras_redirect':        int(len(df_redirect) - 1),  # menos a catch-all
        'n_classes_base_canonica':  int(len(df_base)),
    },
}

with open(RESULTADOS_DIR / 'sumario_pipeline.json', 'w', encoding='utf-8') as f:
    json.dump(sumario, f, indent=2, ensure_ascii=False)

print('=' * 60)
print('SUMÁRIO DO PIPELINE (para o TCC)')
print('=' * 60)
print(json.dumps(sumario, indent=2, ensure_ascii=False))
print(f'\n Salvo em: {RESULTADOS_DIR}/sumario_pipeline.json')

---
## Seção 12 — Próximos passos

Este notebook estabelece o esqueleto funcional. Para fechar o entregável dos 20 dias:

1. **Rodar a ingestão dos 9 PDFs no ChromaDB** (ver notebook `02_ingestao_rag.ipynb` — fora do escopo deste). O código-esqueleto foi fornecido em `arquitetura_rag_chunking.md`.

2. **Fazer fine-tuning do BERTimbau** e salvar em `BASE/modelos/bertimbau_mao_na_roda/`. O notebook vai detectar automaticamente e usar no lugar do baseline.

3. **Ampliar o dataset de teste** de 25 (15%) para 50-80 relatos variados (incluindo relatos novos, não vistos no treino), para números mais robustos no TCC.

4. **Validar com mecânico** as 10 linhas da base canônica (ideal) ou documentar como limitação (se tempo apertar).

5. **Medir latência real em ambiente de produção** — os números aqui são do Colab, que tem variabilidade. Para o TCC, fazer N=100 execuções do mesmo relato e reportar média + desvio.

### Considerações de honestidade metodológica para a defesa

- O dataset de teste é estratificado do mesmo dataset que treinou o classificador. Isso **mede consistência, não generalização real**. Dizer isso explicitamente.
- Relatos que ativam regras de redirect nunca exercitam o classificador — então a acurácia reportada é sobre um subconjunto. Mencionar.
- A base canônica foi escrita por nós (não validada por mecânico), portanto a qualidade da resposta canônica é limitada pelo nosso conhecimento do domínio. Declarar como escopo/limitação.